# Design of the controller's input CS stage's $g_m$ and $c_{iss}$
## Feedback circuit for noise design

In [32]:
import SLiCAP as sl
import sympy as sp
from WLI import updateModel, WLI
from SLiCAPmosNoise import process, specObject, doMOSnoiseDesign

sl.initProject('Hearing loop', notebook=True)

specs     = sl.csv2specs("fb_concept.csv")
spec_dict = sl.specs2dict(specs)
channel   = "P"
IC        = 0.5
    
baseName  = "feedbackConceptNoisyNullor{}18".format(channel.upper())
kicad_sch = "kicad/A1/{}.kicad_sch".format(baseName)
cir       = sl.makeCircuit(kicad_sch)

# Redefine some model equations.
cir, IC_sym, gm_sym = updateModel(cir, IC)

# Assign parameter definitions
sl.specs2circuit(specs, cir)

# Assign a noise budget (Contributions of: source, source termination, feedback network, and input stage)
B_n2      = 0
while B_n2 <= spec_dict["B_n1"].value or B_n2 >= 1:
    B_n2 = float(input("Please enter part of RMS output noise due to source resistance, termination resistance {} < B_n2 < {}"
                 .format(str(sp.N(spec_dict["B_n1"].value,3)), "1"))) 
sl.img2html(baseName + ".svg", 900)

Compiling library: SLiCAP.lib.
Compiling library: SLiCAPmodels.lib.
Creating netlist of kicad/A1/feedbackConceptNoisyNullorP18.kicad_sch using KiCAD
Creating drawing-size SVG and PDF images of kicad/A1/feedbackConceptNoisyNullorP18.kicad_sch
Plotted to '/home/anton/DATA/www/SEDwebsite/_build/SEDwebsite/EE4109-2025-2026/designExample/HearingLoop/Notebooks/img/feedbackConceptNoisyNullorP18.svg'.
Done.
Checking netlist: cir/feedbackConceptNoisyNullorP18.cir
Compiling library: lib/SLiCAP_C18.lib.


Please enter part of RMS output noise due to source resistance, termination resistance 0.400 < B_n2 < 1 0.6


## Show expanded circuit data with parameters: $f$, $g_m$, and $c_{iss}$ 

In [33]:
sl.elementData2html(cir)

RefDes,Nodes,Refs,Model,Param,Symbolic,Numeric
C1,4 0,,C,value,$C_{s}$,$9.382\cdot 10^{-12}$
,,,,vinit,$0$,$0$
C2,out 0,,C,value,$C_{L}$,$10\cdot 10^{-12}$
,,,,vinit,$0$,$0$
C3,out 3,,C,value,$C_{i}$,$2.677\cdot 10^{-9}$
,,,,vinit,$0$,$0$
E2,noise 0 out 0,,E,value,$DIN_{A}$,$\frac{1.872 \cdot 10^{8} f^{4}}{\left(\left(f^{2} + 1.16 \cdot 10^{4}\right) \left(f^{2} + 5.445 \cdot 10^{5}\right)\right)^{0.5} \left(f^{2} + 424.4\right) \left(f^{2} + 1.487 \cdot 10^{8}\right)}$
G1_X1,int_X1 3 1_X1 int_X1,,G,value,$c_{iss X1} s$,$c_{iss} s$
I1_X1,4 3,,I,value,$0$,$0$
,,,,noise,$2 IG q$,$0$


## Determine the spectrum of the weighted output noise as a function of $g_m$ and $c_{iss}$

In [34]:
# Declare symbolic variables
g_m, c_iss, f_T = sp.symbols("g_m, c_iss, f_T")
# Define frequency range
f_min           = spec_dict["f_min"].value
f_max           = spec_dict["f_max"].value
# Calculate the spectral density of the output noise; this may take a while
onoise          = sl.doNoise(cir, pardefs="circuit", numeric=True).onoise
onoise = sl.doNoise(cir, pardefs="circuit", numeric=True).onoise
sl.eqn2html("S_o", onoise, units="V**2/Hz")

## Derive the noise design equation
### Get the output variance in ters of $g_m$ and $c_{iss}$ 

In [35]:
coeffs   = sl.integrated_monomial_coeffs(sp.expand(onoise), (g_m, c_iss), sl.ini.frequency, f_min, f_max)
ovar     = 0
for coeff in coeffs.keys():
    ovar += coeff * coeffs[coeff]
sl.eqn2html("sigma_o^2", ovar, units="V**2")

### Write the output variance in terms of $f_T$ and $c_{iss}$
This equation shows that a high cut-off frequency is benificial to the noise performance

In [36]:
ovar_fT         = sp.N(ovar.subs(g_m, 2 * sp.pi * f_T * c_iss))
sl.eqn2html("sigma_o^2", ovar_fT, units="V**2")

### The noise equation for a specified RMS noise budget
We will define a budget for the contribution of the controller's input stage to the total DIN A weighted output noise.
and then solve $g_m$ after equating this budget with the obtained expression.

In [37]:
var_o = (B_n2 * spec_dict["V_onoise"].value)**2
# noise design equation
gm_ciss = sp.solve(ovar - var_o, g_m)[0]
sl.eqn2html("g_m(c_iss)", gm_ciss, units="S")

## Find the lower limit of $c_{iss}$

In [38]:
num, den  = gm_ciss.as_numer_denom()
c_iss_Min = sp.solve(den, c_iss)[0]
if c_iss_Min <= 0:
    print("ERROR: no valid solution for this noise budget and inversion level!")
else:
    sl.eqn2html("c_iss_Min", c_iss_Min, units= "F")
c_iss_Min

1.82497217492545e-14

## Find the lowest current at the selected inversion coefficient
### Optimum $c_{iss}$

In [39]:
# Lowest current solution at this IC
diff_gm_ciss = sp.diff(gm_ciss, c_iss)
num, den     = diff_gm_ciss.as_numer_denom()
solutions    = sp.solve(num, c_iss)
c_iss_Opt1   = None
c_iss_Lim    = c_iss_Min
for sol in solutions:
    realpart = sp.re(sol)
    imagpart = sp.im(sol)
    if realpart > c_iss_Lim and sp.Abs(imagpart/realpart) < 1e-6:
        c_iss_Lim = realpart
        c_iss_Opt1 = realpart
sl.eqn2html("c_iss_Opt1", c_iss_Opt1, units="F")

### Lowest $g_m$

In [40]:
# Minimum g_m
g_m_Min1 = gm_ciss.subs(c_iss, c_iss_Opt1)
sl.eqn2html("g_m_Min1", g_m_Min1, units="S")

### Determine $W$, $L$ and $I_{DS}$

Please **CHECK** if these values can be applied!

How does the noise budget or the inversion coefficient change $W$, $L$ and $I_{DS}$?

In [41]:
W1, L1, IDS1 = WLI(cir, g_m_Min1, c_iss_Opt1, gm_sym, IC, channel)

In [42]:
sl.eqn2html("W", W1, units="m")

In [43]:
sl.eqn2html("L", L1, units="m")

In [44]:
sl.eqn2html("I_DS", IDS1, units="A")

## Find the lowest $g_m c_{iss}$ product
### Optimum $c_{iss}$

In [45]:
costs           = gm_ciss * c_iss
costs_diff_ciss = sp.diff(costs, c_iss)
num, den        = costs_diff_ciss.as_numer_denom()
solutions       = sp.solve(num, c_iss)
c_iss_Opt2      = None
c_iss_Lim       = c_iss_Min
for sol in solutions:
    realpart    = sp.re(sol)
    imagpart    = sp.im(sol)
    if realpart > c_iss_Lim and sp.Abs(imagpart/realpart) < 1e-6:
        c_iss_Lim  = realpart
        c_iss_Opt2 = realpart
sl.eqn2html("c_iss_Opt2", c_iss_Opt2, units="F")

### Value of $g_m$

In [46]:
g_m_Min2 = gm_ciss.subs(c_iss, c_iss_Opt2)
sl.eqn2html("g_m_Min2", g_m_Min2, units="S")

### Determine $W$, $L$ and $I_{DS}$

Please **CHECK** if these values can be applied!

How does the noise budget or the inversion coefficient change $W$, $L$ and $I_{DS}$?

In [47]:
W2, L2, IDS2 = WLI(cir, g_m_Min2, c_iss_Opt2, gm_sym, IC, channel)

In [48]:
sl.eqn2html("W", W2, units="m")

In [49]:
sl.eqn2html("L", L2, units="m")

In [50]:
sl.eqn2html("I_DS", IDS2, units="A")

## Store the results in the design database

In [51]:
specs.append(sl.specItem(
    "g_m{}1".format(channel.upper()),
    description = "Minimum transconductance {} channel input stage".format(channel.upper()),
    value       = g_m_Min1,
    units       = "S",
    specType    = "design")
)
specs.append(sl.specItem(
    "c_iss{}1".format(channel.upper()),
    description = "c_iss {}-channel input stage at minimum transconductance".format(channel.upper()),
    value       = c_iss_Opt1,
    units       = "F",
    specType    = "design")
)
specs.append(sl.specItem(
    "g_m{}2".format(channel.upper()),
    description = "Transconductance {} channel input stage at minimum g-c product".format(channel.upper()),
    value       = g_m_Min2,
    units       = "S",
    specType    = "design")
)
specs.append(sl.specItem(
    "c_iss{}2".format(channel.upper()),
    description = "c_iss {}-channel input stage at minimum g-c product".format(channel.upper()),
    value       = c_iss_Opt2,
    units       = "F",
    specType    = "design")
)
specs.append(sl.specItem(
    "W_i{}1".format(channel.upper()),
    description = "{}-channel input stage width at minimum transconductance".format(channel.upper()),
    value       = W1,
    units       = "m",
    specType    = "design")
)
specs.append(sl.specItem(
    "L_i{}1".format(channel.upper()),
    description = "{}-channel input stage length at minimum transconductance".format(channel.upper()),
    value       = L1,
    units       = "m",
    specType    = "design")
)
specs.append(sl.specItem(
    "I_i{}1".format(channel.upper()),
    description = "{}-channel input stage current at minimum transconductance".format(channel.upper()),
    value       = IDS1,
    units       = "A",
    specType    = "design")
)
specs.append(sl.specItem(
    "W_i{}2".format(channel.upper()),
    description = "{}-channel input stage width at minimum g-c product".format(channel.upper()),
    value       = W2,
    units       = "m",
    specType    = "design")
)
specs.append(sl.specItem(
    "L_i{}2".format(channel.upper()),
    description = "{}-channel input stage length at minimum g-c product".format(channel.upper()),
    value       = L2,
    units       = "m",
    specType    = "design")
)
specs.append(sl.specItem(
    "I_i{}2".format(channel.upper()),
    description = "{}-channel input stage current at minimum g-c product".format(channel.upper()),
    value       = IDS2,
    units       = "A",
    specType    = "design")
)
sl.specs2csv(specs, "fb_concept.csv")
sl.specs2html(specs, ["design"])

symbol,description,value,units,$B_{i1}$,Relative budget for the current through the feedback network,$0.1$,,$B_{n1}$,"Relative budget for contribution of source, termination and feedback network to the total weighted output noise",$0.4$,,$C_{i}$,Integrator capacitance,$\frac{\tau_{i}}{R_{f}}$,$\mathrm{F}$,$I_{iP1}$,P-channel input stage current at minimum transconductance,$-2.867\cdot 10^{-6}$,$\mathrm{A}$,$I_{iP2}$,P-channel input stage current at minimum g-c product,$-5.783\cdot 10^{-6}$,$\mathrm{A}$,$L_{iP1}$,P-channel input stage length at minimum transconductance,$88.97\cdot 10^{-6}$,$\mathrm{m}$,$L_{iP2}$,P-channel input stage length at minimum g-c product,$217.4\cdot 10^{-9}$,$\mathrm{m}$,$R_{b}$,Bias resistance,$\frac{0.5}{\pi C_{i} f_{min}}$,$\mathrm{\Omega}$,$R_{f}$,Selected value of R_f,$6\cdot 10^{3}$,$\mathrm{\Omega}$,$R_{f max}$,Maximum value of R_f,$6.177\cdot 10^{3}$,$\mathrm{\Omega}$,$R_{f min}$,Minimum value of R_f,$1.393\cdot 10^{3}$,$\mathrm{\Omega}$,$W_{iP1}$,P-channel input stage width at minimum transconductance,$0.003356$,$\mathrm{m}$,$W_{iP2}$,P-channel input stage width at minimum g-c product,$16.55\cdot 10^{-6}$,$\mathrm{m}$,$c_{issP1}$,c_iss P-channel input stage at minimum transconductance,$1.111\cdot 10^{-9}$,$\mathrm{F}$,$c_{issP2}$,c_iss P-channel input stage at minimum g-c product,$36.50\cdot 10^{-15}$,$\mathrm{F}$,$g_{mP1}$,Minimum transconductance P channel input stage,$60.31\cdot 10^{-6}$,$\mathrm{S}$,$g_{mP2}$,Transconductance P channel input stage at minimum g-c product,$121.7\cdot 10^{-6}$,$\mathrm{S}$


## Check the results
We will now check the results with the EKV model without ignoring the lateral velocity saturation and the gate-bulk overlap capacitance.
### Create the circuit and assign parameter values from specifications

In [52]:
cir = sl.makeCircuit(kicad_sch)
sl.specs2circuit(specs, cir)

Creating netlist of kicad/A1/feedbackConceptNoisyNullorP18.kicad_sch using KiCAD
Creating drawing-size SVG and PDF images of kicad/A1/feedbackConceptNoisyNullorP18.kicad_sch
Plotted to '/home/anton/DATA/www/SEDwebsite/_build/SEDwebsite/EE4109-2025-2026/designExample/HearingLoop/Notebooks/img/feedbackConceptNoisyNullorP18.svg'.
Done.
Checking netlist: cir/feedbackConceptNoisyNullorP18.cir
Compiling library: lib/SLiCAP_C18.lib.


### Select values for de MOS device parameters

In [53]:
cir.defPar("W", W2)
cir.defPar("L", L2)
cir.defPar("ID", IDS2)

### Perform noise analysis

In [54]:
noiseResult = sl.doNoise(cir, pardefs="circuit", detector="V_noise", numeric=True)
RMSnoise    = sl.rmsNoise(noiseResult, "onoise", f_min, f_max)
sl.eqn2html("V_noise", RMSnoise, units="V")

# NGspice verification
## Operating point information

In [55]:
sl.img2html("bias{}.svg".format(channel.upper()), 400)

In [56]:
W_max = 50e-6
W     = cir.getParValue("W")
M     = int(W/W_max) + 1
W     = float(W/M)
L     = cir.getParValue("L", numeric=True)
ID    = cir.getParValue("ID", numeric=True)
if channel.upper() == "P":
    ID = -ID
VDS   = 0.45

params = [("ID", ID), ("VDS", VDS), ("W", W), ("L", L), ("M", M)]
simCmd = "OP"
names  = {"V_D"  : "V(d1)",
          "V_G"  : "V(g1)",
          "I_DS" : "@M1[id]",
          "g_m"  : "@M1[gm]",
          "c_iss": "@M1[cgg]"}

fileName = "kicad/Libs/bias{}.kicad_sch".format(channel.upper())
sl.makeCircuit(fileName, language="SPICE")
op_info  = sl.ngspice2traces(sl.ini.cir_path + "bias{}".format(channel.upper()), 
                           simCmd, names, parList=params)

Creating netlist of kicad/Libs/biasP.kicad_sch using KiCAD
Creating drawing-size SVG and PDF images of kicad/Libs/biasP.kicad_sch
Plotted to '/home/anton/DATA/www/SEDwebsite/_build/SEDwebsite/EE4109-2025-2026/designExample/HearingLoop/Notebooks/img/biasP.svg'.
Done.


## Noise verification
### NGspice noisyNullor subcircuit

In [57]:
sl.img2html("noisyNullor{}.svg".format(channel.upper()), 600)

### NGspice DIN A filter subcircuit
For details, please see: **DIN_A.ipynb**

In [58]:
sl.makeCircuit("kicad/A1/DIN_A_SPICE.kicad_sch", language="SPICE")
sl.img2html("DIN_A_SPICE.svg", 1000)

Creating netlist of kicad/A1/DIN_A_SPICE.kicad_sch using KiCAD
Creating drawing-size SVG and PDF images of kicad/A1/DIN_A_SPICE.kicad_sch
Plotted to '/home/anton/DATA/www/SEDwebsite/_build/SEDwebsite/EE4109-2025-2026/designExample/HearingLoop/Notebooks/img/DIN_A_SPICE.svg'.
Done.


In [59]:
sl.img2html("DIN_A_SPICE_MAG.svg", 400)

### NGspice circuit with noisy nullor and DIN A weighting filter

In [60]:
# Define the parameters
params = [("ID", op_info["I_DS"]), ("VDS", op_info["V_D"]), ("W", W), ("L", L), ("M", M), ("VGS", op_info["V_G"])]
cir_par_names = ["R_s", "C_s", "L_s", "R_t", "R_f", "R_b", "C_i", "C_L"]
for par_name in cir_par_names:
    params.append((par_name, cir.getParValue(par_name, numeric=True)))
  
netlist = sl.makeCircuit("kicad/A1/feedbackConceptNoisyNullor{}spice.kicad_sch".format(channel.upper()), language = "SPICE")
sl.img2html("feedbackConceptNoisyNullor{}spice.svg".format(channel.upper()), 700)

Creating netlist of kicad/A1/feedbackConceptNoisyNullorPspice.kicad_sch using KiCAD
Creating drawing-size SVG and PDF images of kicad/A1/feedbackConceptNoisyNullorPspice.kicad_sch
Plotted to '/home/anton/DATA/www/SEDwebsite/_build/SEDwebsite/EE4109-2025-2026/designExample/HearingLoop/Notebooks/img/feedbackConceptNoisyNullorPspice.svg'.
Done.


In [61]:
# Check operating point
sim_cmd = "OP"
names   = {"V_in": "V(in)", "V_fb": "V(fb)", "V_out": "V(out)"}
file_name = "cir/feedbackConceptNoisyNullor{}spice".format(channel.upper())
op_info = sl.ngspice2traces(file_name, sim_cmd, names, parList=params)
sl.eqn2html("V_outDC", op_info["V_out"], units="V")

In [62]:
# Perform noise analysis
sim_cmd = "NOISE V(noise) V1 dec 20 600 6k"
names   = {"v_no": "onoise_total"}
NOISETOT = sl.ngspice2traces(file_name, sim_cmd, names, parList=params, squaredNoise=False)
sl.eqn2html("v_no", NOISETOT["v_no"], units="V")